# Identifying Missing Labels
As we are looking at a small selection of the 50M sources of the potentially 950M we will be going through, some of the labelled data is missing (not from selection, purely from the random sources we've taken).

If missing, will be saved to a new parquet so the training data can be represented.

## Imports

In [1]:
import pandas as pd
import glob

import duckdb

## Loading Labelled CSV

In [2]:
df = pd.read_csv('/media/team_workspaces/AnomalyMatch-IDR1-Search/benchmarking_tests/data/labelled_lenses/labelled_data.csv')

## Doing DuckDB Trick
So, using the DuckDB quick scan of the 50M we can quickly identify the missing SourceIDs from the labelled dataset.

In [5]:
source_ids = list(df.id)

In [6]:
con = duckdb.connect()

# Register the ID list as a table (preserves int64)
con.execute("CREATE TEMP TABLE targets AS SELECT * FROM (VALUES " +
            ",".join(f"({sid})" for sid in source_ids) +
            ") AS t(SourceID)")

missing = con.execute("""
    SELECT t.SourceID
    FROM targets t
    ANTI JOIN read_parquet('/media/team_workspaces/AnomalyMatch-IDR1-Search/benchmarking_tests/data/source_cats/*.parquet') p
        USING (SourceID)
""").df()

missing_ids = missing["SourceID"].tolist()
print(f"{len(missing_ids)} / {len(source_ids)} SourceIDs not found")

58 / 404 SourceIDs not found


## Matching and Getting Sources

In [10]:
con = duckdb.connect()

# Register the ID list as a table (preserves int64)
con.execute("CREATE TEMP TABLE targets AS SELECT * FROM (VALUES " +
            ",".join(f"({sid})" for sid in missing_ids) +
            ") AS t(SourceID)")

result = con.execute("""
    SELECT p.SourceID, p.RA, p.Dec, p.diameter_pixel, p.fits_file_paths
    FROM read_parquet('/media/team_workspaces/AnomalyMatch-IDR1-Search/source_cats_idr1/source_cats_idr1_all_selected/*.parquet') p
    SEMI JOIN targets t USING (SourceID)
""").df()

# result.to_parquet('matched_sources.parquet', index=False)

In [13]:
result.to_parquet('/media/team_workspaces/AnomalyMatch-IDR1-Search/benchmarking_tests/data/source_cats/missing-labels.parquet')